In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from pathlib import Path
from typing import Optional
from itertools import product
from math import ceil

from utils import SECTORS, DR_PRICES, NG_PRICES, ERS, SECTOR_NICE_NAMES, DR_PRICE_NICE_NAMES, NAT_GAS_NICE_NAMES, ER_NICE_NAMES, REGIONS

In [ ]:
# User defined variables
METHOD = "dynamic" # "static" | "dynamic"
REGION = "caiso_cc" # "caiso" | "new_england" | "caiso_cc"
AS_PERCENTAGE = True # display heatmaps as percentage of base case
CMAP_OVERRIDE = None # https://matplotlib.org/stable/users/explain/colors/colormaps.html

def get_scenario_prefix(region: str) -> str:
    if region == "caiso":
        return "c"
    elif region == "new_england":
        return "ne"
    elif region == "caiso_cc":
        return "cc"
    else:
        raise ValueError(f"Invalid region: {region}")

SCENARIOS = [5,10,20,100]

In [3]:
# Path handling (DO NOT CHANGE)
FIGURES_BASE = Path("..","figures","sa")
FIGURES_METHOD = Path(FIGURES_BASE, REGION, METHOD)

In [4]:
# Formatting (DO NOT CHANGE)
OBJ_COST_SCALER = 1 if AS_PERCENTAGE else 1e9
PEAKINESS_SCALER = 1 if AS_PERCENTAGE else 1e3
PEAK_NET_LOAD_SCALER = 1 if AS_PERCENTAGE else 1e3
RAMPING_SCALER = 1 if AS_PERCENTAGE else 1e3
CAPACITY_SCALER = 1 if AS_PERCENTAGE else 1e3
EMISSIONS_SCALER = 1 if AS_PERCENTAGE else 1e6
GENERATION_SCALER = 1 if AS_PERCENTAGE else 1e3

CMAP = CMAP_OVERRIDE if CMAP_OVERRIDE else "coolwarm" if AS_PERCENTAGE else "crest"

OBJ_COST_LABEL = "Deviation from Base (%)" if AS_PERCENTAGE else "($B)"
PEAKINESS_LABEL = "Deviation from Base (%)" if AS_PERCENTAGE else "(GW)"
PEAK_NET_LOAD_LABEL = "Deviation from Base (%)" if AS_PERCENTAGE else "(GW)"
RAMPING_LABEL = "Deviation from Base (%)" if AS_PERCENTAGE else "(GW)"
CAPACITY_LABEL = "Deviation from Base (%)" if AS_PERCENTAGE else "(GW)"
EMISSIONS_LABEL = "Deviation from Base (%)" if AS_PERCENTAGE else "(MMT CO2e)"
GENERATION_LABEL = "Deviation from Base (%)" if AS_PERCENTAGE else "(GWh)"

SHARE_YLABEL = True if AS_PERCENTAGE else False

In [5]:
def get_template_dataframe(include_base: bool = False) -> pd.DataFrame:
    prices = DR_PRICES.copy()
    if include_base:
        prices.insert(0, "None")
    sectors = SECTORS.copy()
    if include_base:
        sectors.insert(0, "None")
    return pd.DataFrame([],
        index=sectors,
        columns=prices
    )
get_template_dataframe(include_base=False)

,high,mid,low,vlow
e,NaN,NaN,NaN,NaN
t,NaN,NaN,NaN,NaN
et,NaN,NaN,NaN,NaN


In [ ]:
def get_sa_scenario_name(
    scenario: str, sector: Optional[str] = None, dr_price: Optional[str] = None
) -> str:
    """Get the scenario name for a given NG price, sector, and DR price"""
    if scenario.startswith("er"):
        dimension = "er"
    elif scenario.endswith("gas"):
        dimension = "ng"
    else:
        raise ValueError(f"Invalid scenario: {scenario}")

    if dimension == "er":
        assert scenario in ERS, f"Invalid ER: {scenario}. Expected one of {ERS}"
    elif dimension == "ng":
        gas_price = scenario.split("-")[-1]
        assert gas_price in NG_PRICES, (
            f"Invalid NG price: {scenario}. Expected one of {NG_PRICES}"
        )
    else:
        raise ValueError(f"Invalid scenario: {scenario}")

    if not (sector or dr_price):
        return scenario
    assert sector and dr_price, "Sector and DR price must be provided"
    assert sector in SECTORS, f"Invalid sector: {sector}. Expected one of {SECTORS}"
    assert dr_price in DR_PRICES, (
        f"Invalid DR price: {dr_price}. Expected one of {DR_PRICES}"
    )
    return f"{sector}-{dr_price}-{scenario}"

In [ ]:
def get_sa_datapoint(
    region: str,
    scenario: str,
    result: str,
    method: Optional[str] = None,
    sector: Optional[str] = None,
    dr_price: Optional[str] = None,
) -> pd.DataFrame:
    """Get the datapoint for a given NG price, sector, and DR price"""
    assert region in REGIONS, f"Invalid region: {region}. Expected one of {REGIONS}"
    scenario = get_sa_scenario_name(scenario, sector, dr_price)

    if method:
        assert method in METHODS, f"Invalid method: {method}. Expected one of {METHODS}"
        assert sector and dr_price, "Sector and DR price must be provided"
        p = Path(
            DATA_DIR,
            region,
            "processed",
            method,
            scenario,
            "datapoint",
            f"{result}.csv",
        )
    else:
        p = Path(DATA_DIR, region, "processed", scenario, "datapoint", f"{result}.csv")

    assert p.exists(), (
        f"Data point not found for:\nRegion={region}\nScenario={scenario}\nSector={sector}\nDr_Price={dr_price}\nMethod={method}\nResult={result}"
    )

    return pd.read_csv(p, index_col=0)

In [ ]:
def get_heatmap_data(result: str, metric: str, as_percentage: bool = False) -> dict[str, pd.DataFrame]:
    data = {}
    scenarios = SCENARIOS
    for scenario in scenarios:
        include_base = False if as_percentage else True
        hm = get_template_dataframe(include_base=include_base)
        for sector, dr_price in product(SECTORS, DR_PRICES):
            df = get_sa_datapoint(
                region=REGION,
                scenario=scenario,
                result=result,
                sector=sector,
                dr_price=dr_price,
                method=METHOD
            )
            datapoint = float(df.at[metric, "value"])
            hm.at[sector, dr_price] = datapoint
        df = get_datapoint(region=REGION, scenario=scenario, result=result)
        datapoint = float(df.at[metric, "value"])
        if as_percentage:
            if datapoint != 0:
                hm = (hm - datapoint).div(datapoint).mul(100).round(2)
            elif hm.sum().sum() <= 1e-3:
                pass
            else:
                raise ValueError(f"datapoint = {datapoint} | hm = {hm}")
        else:
            datapoint = df.at[metric, "value"]
            hm.at["None", "None"] = datapoint
        data[scenario] = hm.astype(float)
    return data

In [ ]:
get_heatmap_data("cost", "objective_adj", as_percentage=AS_PERCENTAGE)

In [7]:
def generate_heatmap(data: dict[str, pd.DataFrame], scaler: Optional[float] = None, **kwargs) -> tuple[plt.figure, plt.axes]:
    
    if not scaler:
        scaler = 1
    
    ncols = 2
    nrows = ceil(3 / ncols)
    
    ylabel = kwargs.get("ylabel", "")
    
    fig, axs = plt.subplots(nrows, ncols, figsize=(12, 5 * nrows))
    
    row = 0
    col = 0
    
    title = kwargs.get("title", "")
    if title:
        fig.suptitle(title, fontsize=20)
        
    custom_title = kwargs.get("custom_title", "")
    
    for scenario, df in data.items():
        
        df = df.T.rename(index=DR_PRICE_NICE_NAMES).rename(columns=SECTOR_NICE_NAMES)
        
        if custom_title:
            title = custom_title
        elif SCENARIO == "ng":
            title = NAT_GAS_NICE_NAMES[scenario]
        elif SCENARIO == "er":
            title = ER_NICE_NAMES[scenario]
        else:
            raise ValueError(f"Invalid scenario: {SCENARIO}")
        
        df = df.div(scaler).round(2)
        
        hm = sns.heatmap(df, annot=True, cmap="crest", ax=axs[row, col], fmt=".1f")
        
        if ylabel:
            colorbar = hm.collections[0].colorbar
            colorbar.ax.set_ylabel(ylabel, rotation=0, labelpad=10, y=-0.05)
        
        axs[row, col].set_title(title, fontsize=14)
        axs[row, col].tick_params(axis="y", labelrotation=0)
        
        col += 1
        if col > 1:
            col = 0
            row += 1
        
    if col != 0:
        axs[row, 1].axis('off')
        
    plt.tight_layout(rect=[0, 0, 1, 0.99])
        
    return fig, axs

In [8]:
def generate_heatmap_3col(data: dict[str, pd.DataFrame], scaler: Optional[float] = None, share_ylabel: Optional[bool] = False, **kwargs) -> tuple[plt.figure, plt.axes]:
    
    if not scaler:
        scaler = 1
    
    ncols = 3
    nrows = 1
    
    fig, axs = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))

    col = 0
    
    title = kwargs.get("title", "")
    if title:
        fig.suptitle(title, fontsize=20)
        
    custom_title = kwargs.get("custom_title", "")
    
    ylabel = kwargs.get("ylabel", "")
    
    cmap = kwargs.get("cmap", "viridis")
    
    data = {x: y.fillna(0) for x, y in data.items()}
    
    if share_ylabel:
        all_values = pd.concat(
            [df.T.rename(index=DR_PRICE_NICE_NAMES).rename(columns=SECTOR_NICE_NAMES).div(scaler).stack()
             for df in data.values()]
        )
        vmin, vmax = all_values.min(), all_values.max()

        # force range to be symmetric around 0 so center always stays at 0
        abs_max = max(abs(vmin), abs(vmax))
        vmin, vmax = -abs_max, abs_max
        
        if vmin == vmax:
            vmin -= 1
            vmax += 1
            
        norm = TwoSlopeNorm(vcenter=0, vmin=vmin, vmax=vmax)
    else:
        vmin = vmax = None
        norm = None
    
    heatmaps = [] # for shared colorbar
    
    for scenario, df in data.items():
        
        df = df.T.rename(index=DR_PRICE_NICE_NAMES).rename(columns=SECTOR_NICE_NAMES)
        
        if custom_title:
            title = custom_title
        elif SCENARIO == "ng":
            title = NAT_GAS_NICE_NAMES[scenario]
        elif SCENARIO == "er":
            title = ER_NICE_NAMES[scenario]
        else:
            raise ValueError(f"Invalid scenario: {SCENARIO}")
        
        df = df.div(scaler).round(2)
        
        if col == 0:
            yticklabels = True
        else:
            yticklabels = False
            
        hm = sns.heatmap(
            df, 
            annot=True, 
            cmap=cmap,
            cbar=False if share_ylabel else True,
            ax=axs[col], 
            fmt=".1f", 
            yticklabels=yticklabels,
            annot_kws={
                "fontsize":"x-large"
            },
            vmin=vmin,
            vmax=vmax,
            norm=norm
        )
        heatmaps.append(hm)

        if not share_ylabel:
            colorbar = hm.collections[0].colorbar
            colorbar.ax.tick_params(labelsize=14)
            if ylabel:
                colorbar.ax.set_ylabel(ylabel, rotation=90, labelpad=10, y=0.5, fontsize=16)
        
        axs[col].set_title(title, fontsize=18, pad=20)
        
        axs[col].tick_params(axis="y", labelrotation=0, labelsize=14)
        axs[col].tick_params(axis="x", labelrotation=45, labelsize=14)
        
        col += 1
        
    # fig.supxlabel("Demand Response Sector", fontsize =18)
    fig.supylabel("Demand Response Cost", fontsize =18)
        
    fig.tight_layout(rect=[0.02, 0, 0.80, 1])
        
    if share_ylabel:
        cbar = fig.colorbar(
            heatmaps[0].collections[0],
            ax=axs,
            orientation="vertical",
            fraction=0.02,
            pad=0.02,
            norm=norm
        )
        cbar.ax.tick_params(labelsize=14)
        if ylabel:
            cbar.ax.set_ylabel(ylabel, rotation=90, labelpad=10, y=0.5, fontsize=16)
            
        
        
    return fig, axs

In [ ]:
data = get_heatmap_data("cost", "objective_adj", as_percentage=AS_PERCENTAGE)
# fig, _ = generate_heatmap_3col(data, scaler=OBJ_COST_SCALER, title="", ylabel=OBJ_COST_LABEL, share_ylabel=True, cmap=CMAP)
# save_f = Path(FIGURES_METHOD, "objective_adj.png")
# save_f.parent.mkdir(parents=True, exist_ok=True)
# fig.savefig(save_f, dpi=300, bbox_inches="tight")